# Error Analysis\n\nLoad predictions from `results/predictions.csv` (produced by `evaluate.py`) and run full error analysis with visualizations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score,
    accuracy_score, f1_score,
)
from sklearn.calibration import calibration_curve

sns.set_theme(style="whitegrid")

df = pd.read_csv("../results/predictions.csv")
df.head()

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, group in df.groupby("detector"):
    labels = group["label"].values
    scores = group["score"].values
    if len(set(labels)) < 2:
        continue
    fpr_vals, tpr_vals, _ = roc_curve(labels, scores)
    auc_val = roc_auc_score(labels, scores)
    ax.plot(fpr_vals, tpr_vals, label=f"{name} (AUC={auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Detectors")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## Confusion Matrices

In [ ]:
detectors = df["detector"].unique()
fig, axes = plt.subplots(1, len(detectors), figsize=(5 * len(detectors), 4))

if len(detectors) == 1:
    axes = [axes]

for ax, name in zip(axes, detectors):
    group = df[df["detector"] == name]
    preds = (group["score"] > 0.5).astype(int)
    cm = confusion_matrix(group["label"], preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Human", "AI"], yticklabels=["Human", "AI"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(name)

fig.suptitle("Confusion Matrices", y=1.02)
fig.tight_layout()
plt.show()

## Per-Generator Accuracy

In [ ]:
for name in detectors:
    group = df[df["detector"] == name]
    group = group.copy()
    group["correct"] = ((group["score"] > 0.5).astype(int) == group["label"]).astype(int)

    source_acc = group.groupby("source")["correct"].mean().sort_values()

    fig, ax = plt.subplots(figsize=(10, max(3, len(source_acc) * 0.4)))
    ax.barh(source_acc.index, source_acc.values, color="steelblue")
    ax.set_xlabel("Accuracy")
    ax.set_title(f"Per-Generator Accuracy — {name}")
    ax.set_xlim(0, 1)
    fig.tight_layout()
    plt.show()

## Accuracy by Text Length

In [ ]:
for name in detectors:
    group = df[df["detector"] == name].copy()
    group["word_count"] = group["text"].str.split().str.len()
    group["correct"] = ((group["score"] > 0.5).astype(int) == group["label"]).astype(int)
    group["length_bucket"] = pd.cut(
        group["word_count"],
        bins=[0, 50, 100, 200, 500, float("inf")],
        labels=["<50", "50-100", "100-200", "200-500", "500+"],
    )

    bucket_acc = group.groupby("length_bucket", observed=True)["correct"].mean()

    fig, ax = plt.subplots(figsize=(8, 5))
    bucket_acc.plot(kind="bar", ax=ax, color="teal")
    ax.set_xlabel("Text Length (words)")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"Accuracy by Text Length — {name}")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=45)
    fig.tight_layout()
    plt.show()

## Threshold Sensitivity

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.05)

for name in detectors:
    group = df[df["detector"] == name]
    labels = group["label"].values
    scores = group["score"].values

    accuracies = [accuracy_score(labels, (scores > t).astype(int)) for t in thresholds]
    f1s = [f1_score(labels, (scores > t).astype(int), zero_division=0) for t in thresholds]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(thresholds, accuracies, label="Accuracy", marker="o", markersize=3)
    ax.plot(thresholds, f1s, label="F1", marker="s", markersize=3)
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Score")
    ax.set_title(f"Threshold Sensitivity — {name}")
    ax.legend()
    fig.tight_layout()
    plt.show()

## Calibration Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

for name in detectors:
    group = df[df["detector"] == name]
    labels = group["label"].values
    scores = group["score"].values
    if len(set(labels)) < 2:
        continue
    prob_true, prob_pred = calibration_curve(labels, scores, n_bins=10, strategy="uniform")
    ax.plot(prob_pred, prob_true, marker="o", label=name)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfectly calibrated")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration — All Detectors")
ax.legend()
fig.tight_layout()
plt.show()

## Failure Case Analysis

In [ ]:
for name in detectors:
    group = df[df["detector"] == name].copy()
    group["pred"] = (group["score"] > 0.5).astype(int)

    fp = group[(group["pred"] == 1) & (group["label"] == 0)].nlargest(5, "score")
    fn = group[(group["pred"] == 0) & (group["label"] == 1)].nsmallest(5, "score")

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"\nFalse Positives: {((group['pred'] == 1) & (group['label'] == 0)).sum()}")
    print(f"False Negatives: {((group['pred'] == 0) & (group['label'] == 1)).sum()}")

    print(f"\n--- Worst False Positives (human misclassified as AI) ---")
    for _, row in fp.iterrows():
        print(f"  Score: {row['score']:.4f} | Source: {row['source']}")
        print(f"  Text: {row['text'][:150]}...\n")

    print(f"--- Worst False Negatives (AI misclassified as human) ---")
    for _, row in fn.iterrows():
        print(f"  Score: {row['score']:.4f} | Source: {row['source']}")
        print(f"  Text: {row['text'][:150]}...\n")

## Score Distribution

In [ ]:
fig, axes = plt.subplots(1, len(detectors), figsize=(5 * len(detectors), 4), sharey=True)

if len(detectors) == 1:
    axes = [axes]

for ax, name in zip(axes, detectors):
    group = df[df["detector"] == name]
    human = group[group["label"] == 0]["score"]
    ai = group[group["label"] == 1]["score"]

    ax.hist(human, bins=30, alpha=0.6, label="Human", color="green")
    ax.hist(ai, bins=30, alpha=0.6, label="AI", color="red")
    ax.axvline(0.5, color="black", linestyle="--", alpha=0.5)
    ax.set_xlabel("Score")
    ax.set_title(name)
    ax.legend()

axes[0].set_ylabel("Count")
fig.suptitle("Score Distribution by Class", y=1.02)
fig.tight_layout()
plt.show()